In [34]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt

--2026-07-10 23:57:45--  https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M  --.-KB/s    in 0.01s   

2026-07-10 23:57:46 (111 MB/s) - ‘input.txt.2’ saved [1115394/1115394]



In [35]:
dataset = open('input.txt', 'r', encoding='utf-8')
text = dataset.read()

In [36]:
print(len(text))

1115394


In [37]:
print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [38]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars)) #the first element is actually '\n' and not ' ' like andrej said
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [39]:
encode_map = {ch:i for i,ch in enumerate(chars)} #maps characters to integers
decode_map = {i:ch for i,ch in enumerate(chars)} #maps integers back to characters

encode = lambda s: [encode_map[c] for c in s] #assigns each character of the string 's' to an int from the map 'encode_map'
decode = lambda l: ''.join(decode_map[c] for c in l) #gets the corresponding character for every integer in the list created by the 'encode_map' for the string

In [40]:
print(encode('hello world'))
print(decode(encode('hello world'))) # first converting string to list of integers and then mapping back to characters joining them together

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


In [41]:
import torch
data = torch.tensor(encode(text), dtype=torch.long) # coverting all of the input.txt into encoded list of integers for each character. torch.long makes sure it is stored as integers.
print(data.shape,data.dtype)
print(data[:500])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [42]:
n = int(0.9* len(data))
train_data = data[:n]
val_data = data[n:]

In [43]:
torch.manual_seed(1337) # to get same numbers as andrej (helps in croschecking)

batch_size = 4
block_size = 8

def get_batch(split_type):
  data = train_data if split_type == 'train' else val_data
  ix = torch.randint(len(data)-block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x,y

xb, yb = get_batch('train')
print(xb.shape, '\n', xb)
print(yb.shape, '\n', yb)

torch.Size([4, 8]) 
 tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
torch.Size([4, 8]) 
 tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [44]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337) # to keep track with Andrej

class BigramLM(nn.Module):

  def __init__(self,vocab_size):
    super().__init__() # essentially to inherit the __init__() of the superclass (nn.Module)
    self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

  def forward(self, i, targets=None):
    logits = self.token_embedding_table(i) # (Batch, Time, Channel)
    if targets == None:
      loss = None
    else:
      B,T,C= logits.shape
      logits = logits.view(B*T,C)#3D to 2D compresses (4,8,65) to (32,65)
      targets = targets.view(B*T) #2D to 1D compresses (4,8) to (32,)
      loss = F.cross_entropy(logits,targets)
    return logits , loss

  def generate(self,i,max_new_tokens):
    for _ in range(max_new_tokens):
      logits, loss = self(i) #runs forward() on each i in the (4,8) xb
      logits = logits[:,-1,:] #takes only the last character of each of the 4 rows so shape is now (4,65)
      probs = F.softmax(logits,dim=-1) #converts logits to probabilities
      i_next = torch.multinomial(probs, num_samples = 1) #randomly picks one number (from the 65 available) weighted by the probabilities
      #i_next = torch.argmax(probs, dim=-1, keepdim=True)  # instead of torch.multinomial(probs, num_samples=1)
      i = torch.cat((i,i_next), dim=1) # concatenates the 4 new i_next (shape is (4,1)) to the existing xb (shape of xb goes from 4,8 to 4,9 and so on every iteration)
    return i

model = BigramLM(vocab_size)
logits, loss = model(xb,yb)
print(loss)

tensor(4.8786, grad_fn=<NllLossBackward0>)


In [45]:
idx = torch.zeros((1,1), dtype = torch.long)
print(decode(model.generate(idx,100)[0].tolist()))


SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [46]:
optimizer = torch.optim.AdamW(model.parameters(),lr=0.001)

In [47]:
batch_size = 32
for steps in range(10000):
  xb,yb = get_batch('train')
  logits,loss = model(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

print(loss.item())

2.382369041442871


In [48]:
print(decode(model.generate(i = torch.zeros((1,1), dtype = torch.long),max_new_tokens=500)[0].tolist()))


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecorro llaus a!
OLeneerithesinthengove fal amas trr
TI ar I t, mes, n IUSt my w, fredeeyove
THek' merer, dd
We ntem lud engitheso; cer ize helorowaginte the?
Thak orblyoruldvicee chot, p,
Bealivolde Th li


In [49]:
# print("lets play hangman")
# import random
# bank = ['python','computer','string','loop']
# answer = random.choice(bank)

# display = []
# for _ in range(len(answer)):
#   display.append('_ ')

# print(''.join(display))

# def play(answer,display):
#   guessed = set()
#   lives = 6
#   while '_ ' in display:
#     if lives == 0:
#       print('out of lives')
#       break
#     else:
#       guess = input("guess a letter: ").lower()
#       if len(guess) != 1 or not guess.isalpha(): print("invalid input")
#       elif guess in guessed: print('already guessed')
#       else:
#         if guess in answer:
#           guessed.add(guess)
#           print('nice')
#         else:
#           lives -= 1
#           print(f'letter not found. lives left: {lives}')

#         if guess in answer:
#           idx = [i for i, ch in enumerate(answer) if ch == guess]
#           for i in idx:
#             display[i] = guess
#         print(''.join(display))

# play(answer,display)

In [52]:
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape #shape (4,8,2)

torch.Size([4, 8, 2])

##Version 1

In [56]:
xbow = torch.zeros((B,T,C)) # bow = bag of words, since we are averaging all of the context prior to the current element.
for b in range(B):
  for t in range(T):
    xprev = x[b,:t+1] #drops the b'th dimention so shape is t,c for each xprev. takes all pairs (since c is 2) upto t'th term for the current batch.
    xbow[b,t] = torch.mean(xprev,0) # compresses the xprev along 0th position to form a (2,) shape that replaces the zeros at b,t in xbow.

##Version 2

In [83]:
w = torch.tril(torch.ones(T,T)) # tril creates a lower triangular matrix filled with ones in the diagonal and below. creates a 8x8 lower triangular matrix
w = w / w.sum(1, keepdim=True) # dividing each 1 by summ of the total ones in that row. essentially dividing the whole row into equal elements summing to one instead of the ones.
xbow2 = w @ x # matrix multiplication faster multiplying w (shaped T,T) and x (shaped B,T,C). since they are un equal, pytorch creates a B dimension so essentially replicates the T,T matrix B times to be able to multiply to x. This does (B,T,T) @ (B,T,C) = (B,T,C) matrix

##Version 3

In [84]:
tril = torch.tril(torch.ones(T,T))# lower triangular matrix of ones
w = torch.zeros(T,T) #init w matrix with all  (this is not zeros in practice)
w = w.masked_fill(tril == 0,float('-inf')) # fills matrix w with float('-inf') wherever the mask tril==0 is true (so upper triangle excluding diagonal) think of it like the future tokens cannot communicate with the past
w = F.softmax(w,dim=-1) #softmax converts raw numbers into probabilities by row
xbow3 = w @ x